In [0]:

# Databricks notebook source

from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

CATALOG = "adbsmartdatatdp"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"USE CATALOG {CATALOG}")

# CUSTOMERS
customers = spark.table(f"{CATALOG}.{BRONZE}.customers_raw")
customers = (
    customers.dropDuplicates(["customer_id"])
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    .select(
        col("customer_id").cast("int"),
        "full_name","email","phone","city","country",
        col("registration_date").cast("date")
    )
)
customers.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{CATALOG}.{SILVER}.customers")

# PRODUCTS
products = spark.table(f"{CATALOG}.{BRONZE}.products_raw")
products = (
    products.filter(col("price")>0)
    .filter(col("stock")>=0)
    .dropDuplicates(["product_id"])
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("brand", initcap(trim(col("brand"))))
    .withColumn("price", col("price").cast(DecimalType(10,2)))
    .select(
        col("product_id").cast("int"),
        "product_name","category","brand","price"
    )
)
products.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{CATALOG}.{SILVER}.products")

# BRANCHES
branches = spark.table(f"{CATALOG}.{BRONZE}.branches_raw")
branches = (
    branches.dropDuplicates(["branch_id"])
    .withColumn("branch_name", initcap(trim(col("branch_name"))))
    .withColumn("city", initcap(trim(col("city"))))
    .select(col("branch_id").cast("int"),"branch_name","city","country")
)
branches.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{CATALOG}.{SILVER}.branches")

# EMPLOYEES
employees = spark.table(f"{CATALOG}.{BRONZE}.employees_raw")
employees = (
    employees.dropDuplicates(["employee_id"])
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    .select(col("employee_id").cast("int"),"full_name","position",col("branch_id").cast("int"))
)
employees.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{CATALOG}.{SILVER}.employees")

# SALES
sales = spark.table(f"{CATALOG}.{BRONZE}.sales_raw")
sales = (
    sales.filter(col("quantity")>0)
    .filter(col("unit_price")>0)
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10,2)))
    .withColumn("discount", col("discount").cast(DecimalType(5,2)))
    .withColumn("subtotal",(col("quantity")*col("unit_price")).cast(DecimalType(12,2)))
    .withColumn("total",(col("subtotal")-(col("subtotal")*col("discount"))).cast(DecimalType(12,2)))
    .select(
        col("sale_id").cast("int"),
        col("customer_id").cast("int"),
        col("product_id").cast("int"),
        col("sale_date").cast("date"),
        col("quantity").cast("int"),
        "unit_price","discount","subtotal","total"
    )
)
sales.write.mode("overwrite").option("overwriteSchema","true").format("delta").saveAsTable(f"{CATALOG}.{SILVER}.sales")

print("SILVER LOAD COMPLETED SUCCESSFULLY")
